### Working with data in PySpark

Here, the data cleaning and preparation means to, 
1. fix missing values
2. fix wrong data types
3. remove duplicates
4. handle outliers 
5. standardise formats
6. prepare data for model/analysis 

#### 1. Handling Missing Values

In [0]:
# dropping nulls
df.dropna()

In [0]:
# fill nulls
df.fillna()
df.fillna({"age":0, "salary":3000})

#### 2. Removing Duplicates:- 
same row appears multiple time

In [0]:
df.dropDuplicates() # for all
df.dropDupicates(["email"]) # for specific column 

#### 3. Fixing Data Types

(i) Fixing the datatypes, check first

In [0]:
df.printSchema()
# to hv an idea abt datatypes of all columns

(ii) fix of this 

In [0]:
df.withColumn("age", col("age").cast("int"))

#### 4. Renaming Columns

In [0]:
df.withColumnRenamed("Customer Name", "Customer_Name")

#### 5. Filtering Bad Data

- need to remove negative ages.
- need to remove impossible ages. 

In [0]:
df.filter(df.age>0)

#### 6. Handling Outliers

ex.- Salary = 1000000, (avg = 50000)

In [0]:
df.filter(df.salary < 1000000)

#### 7. Standardising Data

ex - "male", "Male", "M"

In [0]:
from pyspark.sql.functions import lower

In [0]:
df.withColumn("gender", lower(df.gender))

#### 8. Working with Data

ex - "25-03-2026", "03-25-2026"

In [0]:
df = df.withColumn("date", to_date("date", "dd-mm-yyyy"))

#### 9. Creating new columns 

In [0]:
from pyspark.sql.functions import col

In [0]:
df - df.withColumn("bonus", col("salary")*0.1)

#### 10.Dropping Unnecessary Columns

In [0]:
df.drop("Unwanted_Column")

#### Handling Missing Values

*1. Dropping Missing Values*

In [0]:
df.dropna()
df.dropna(how = "any")
df.dropna(how = "all")
dr.dropna(subset = ["age"])

*2. Filling Missing Values*

In [0]:
df.fillna(0)

In [0]:
df.fillna({
    "age" : 25,
    "salary":3000
})

**Using Mean/Median to fill**

In [0]:
from pyspark.sql.functions import mean

In [0]:
mean_value = df.select(mean("salary")).collect()[0][0]
df = df.fillna({"salary": mean_value})

*3. Filling Forward and Backward Fill(Time - Series)* 

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import last

In [0]:
window = Window.orderBy("date").rowsBetween(-1, 0)
df = df.withColumn(
                    "filled value",
                    last("value", ignoreNulls = True)
                    .over(window)
)

*4. Conditional Filling*

In [0]:
from pyspark.sql.functions import *

In [0]:
df = df.withColumn(
                    "age",
                    when (col("age").isNull(), 25)
                    .otherwise(col("age"))
)

*5. Flag Missing Values*

In [0]:
from pyspark.sql.functions import col

In [0]:
df = df.withColumn(
                "salary_missing",
                col("salary").isNull()
)

*6. Replacing Specific Values*

In [0]:
df.replace("NA", None)

In [0]:
df.replace(" ", None)

### Data Normalisation and Transformations

- Normalisations:- scalling numbers - making values comparable. 

- Transformation:- changing structure/format - making data usable. 

#### DATA NORMALISATION

ex:- 
a. salary:- 10,000 - 1,00,000

b. age:- 18 - 60 

In this model,  will prioritize salary just coz it's bigger than others. \
this is technically wrong. 

**Ways to Normalize**

**1. Min - max scaling**

formula:- (x-min)/(max-min)

In [0]:
from pyspark.ml.features import MinMaxScaler, VectorAssembler
assembler = VectorAssembler(inputCols = ["Age"], outputCol = "age_vec")
df = assembler.transform(df)
Scaler = MinMaxScaler(inputCol = "age_vec", outputCol = "age_scaled")
df = Scaler.fit(df).transform(df)

**2. Standardisation (z-score)**

In [0]:
from pyspark.ml.feature import StandardScaler
scaler = StandardScaler(inputCol = "age_vec", outputCol = "age_scaled")
df = scaler.fit(df).transform(df)

#### DATA TRANSFORMATION

**1. Column Transformation**

In [0]:
from pyspark.sql.functions import col
df = df.withColumn("bonus", col("salary")*0.1)

**2. String Transformation**

In [0]:
from pyspark.sql.functions import lower, upper, trim

In [0]:
df = df.withColumn("name", lower(trim(col("name"))))
df = df.withColumn("name", upper(trim(col("name"))))

**3. Encoding Categorical Data**

In [0]:
from pyspark.ml.feature import StringIndexer
indexer = StringIndexer(inputCol = "gender", outputCol = "gender_index")
df = indexer.fit(df).transform(df)

In [0]:
from pyspark.ml.feature import OneHotEncoder
encoder = OneHotEncoder(inputCol = "gender_index", outputCol = "gender_vec")
df = encoder.fit(df).transform(df)

**4. Aggregation Transformation**

In [0]:
df.groupBy("department").sum("salary")

**5. Pivoting Data**

In [0]:
df.groupBy("department").pivot("gender").count()

**6. Exploding Arrays**

In [0]:
from pyspark.sql.functions import explode
df.select(explode(df.skills)).show()
## complex data to flat structure

**7. Casting & Type Conversion**

In [0]:
df = df.withColumn("salary", col("salary").cast("double"))

### Working with Categorical Data

**Categorical Data**
1. Nominal - No Order
2. Ordinal - Order

**Step - 1** \
**Clean Categories first**

ex:- "Male", "male", "m", "MALE"

In [0]:
from pyspark.sql.functions import lower, trim

In [0]:
df  = df.withColumn("gender", lower(trim(df.gender)))

**Step - 2** \
**Check Unique Values**

must know your categories before encoding

In [0]:
df.select("gender").distinct().show()

**Step - 3** \
**String Indexes**

In [0]:
from pyspark.ml.feature import StringIndexer

In [0]:
indexer = StringIndexer(inputCol = "gender", outputCol = "gender_index")
df = indexer.fit(df).transform(df)

**Step - 4**\
**One - Hot Encoding**

In [0]:
from pyspark.ml.feature import OneHotEncoder

In [0]:
encoder = OneHotEncoder(inputCol = "gender_index", outputCol = "gender_vec")
df = encoder.fit(df).transform(df)

**Step - 5**\
**Handle high Cardinality**

1. Group Rare Categories

In [0]:
top_categories = ["delhi", "mumbai"]
df = df.withColumn(
                    "city_clean"
                    when (df.city_isin(top_categories), 
                          df.city).otherwise("other")
)

2. Frequency Encoding

In [0]:
top_count = ["delhi", "mumbai"]
df = df.withColumn(
                    "city_clean"
                    when (df.city_isin(top_count), 
                          df.city).otherwise("other")
)

**Step - 6**\
**Ordinal Encoding**

ex:- low->1, med->2, high->3

In [0]:
from pyspark.sql.functions import when
df = df.withColumn(
                    "risk_level"
                    when(df.risk == "low", 1)
                    .when(df.risk == "med", 2)
                    ,when(df.risk == "high", 3)
)

**Step - 7**\
**Handle Missing Categories**


In [0]:
df = df.fillna({"gender":"unknown"})

### Performance Tuning in PySpark

when we write,

#### 1. Understanding Jobs, Stage and Tasks

In [0]:
df = spark.read.csv("data.csv")
df.filter("age>25").groupBy("city").count().show()

Here, \
a job = a action \
df.show() - job - 1\
df.count() - job - 2\
df.filter - job - 3 \

Here, each action creates a separate job. 

Here, we have a shuffle, will split \
df.filter("age>25") - filter -> no shuffle -> stage - 1\
df.groupBy("city") -> groupBy -> shuffle -> stage -2\
df.count()\

groupBy(), join(), distinct(), orderBy() \
this causes shuffle. 

task = smallest unit of work, \
df  = no. of partitions = no. of task/stage. 

At last,
- 1 job  = show() 
- read + filter = stage - 1 (no shuffle)
- join  = stage - 2(shuffle)
- groupBy = stage - 3 (shuffle)

#### 2. Tungsten Execution Engine

Apache Spark Tungsten is spark's low - level execution engine designed to - \
a. use CPU efficiently\
b. optimise memory + cache usage

Tungsten is there -> \
uses heap - off memory -> \
as it stores data in binary format.

by using it, 
results in, 
- faster processing 
- manages memory manually. 

Tungsten stores data in compact format, ensures better CPU cache locally. 

 - RDD = no tungsten optimisation
 - dataframe = full tungsten + catalyst optimization

### Memory Management & Garbage Collection 

Spark uses unified memory manager(too big buckets) which splits memory into two parts. 

**Types of Memory**
1. Execution Memory
2. Storage Memory

1. Execution Memory:-
 used for
  - shuffles
  - joins
  - sorts
  - aggregations

-- it's basically a temporary memory. 

2. Storage Memory:- used for
 - cache
 - persisted data

-- long term holding memory 

**Note**
1. if execution needs more memory -> steals from storage.
2. storage full -> cache data is evicted. 

#### Garbage Collection

1. removes unused java objects.
2. frees heap memory
3. pauses execution


Garbage collection becomes problem:- \
        1. bad memory tuning. \
        2. large datasets in heap.

**CASE - 1:- \
Job crashes -> out of memory(oom)**

Reasons:- 
1. too many partitions -> too many tasks -> memory pressure.
2. large shuffle.
3. too much cache.

**CASE - 2:-
Cache, but still slow**

Reasons:- 
1. data got evicted from storage memory

**CASE - 3
Job slow manually**

Reasons:-
1. heavy garbage collection(gc) style.
2. memory thrashing between execution and storage. 

too manage GC:-
1. central partitions
2. cache selectivity
3. shuffle size(looked)

#### Tuning Parallelism

Parallelism:-
how many task run at the same time. 
